In [ ]:
import os
import pymongo
import pandas as pd
from dotenv import load_dotenv
from collections import Counter
import ipywidgets as widgets
from IPython.display import display, clear_output # Import clear_output for interactive widgets

# Load environment variables from .env file

#return client, db

# Connect to the database
client, db = connect_to_mongodb()

collection = db['student_gap_qc'] # Assuming the collection name is 'student_gap_qc'
data = list(collection.find({}))
df_student_gap_qc = pd.DataFrame(data)

print(f"Successfully loaded {len(df_student_gap_qc)} documents into df_student_gap_qc.")

# Assuming df_student_gap_qc is already loaded globally by previous cells.
# We are reverting to use df_student_gap_qc directly, without school merge.

# Initialize dictionaries (global scope for analysis function) to store aggregated results
all_chapters_student_final_batch_labels = {}
all_chapters_students_by_batch_label = {}
# New global dictionary to store detailed QC breakdown for all students, indexed by chapter and book
all_students_qc_breakdown = {}
# New global dictionaries for weak skills and combined groupings
all_chapters_student_qc_label_and_weak_skills = {}
all_chapters_grouped_students_by_weak_skills_and_qc = {}


# --- Helper functions to get dropdown options (Revised for Grade -> Book -> Chapter order) ---
def get_unique_grades():
    # Ensure df_student_gap_qc is available and has 'gradeName'
    if 'df_student_gap_qc' in globals() and not df_student_gap_qc.empty and 'gradeName' in df_student_gap_qc.columns:
        return sorted(df_student_gap_qc['gradeName'].unique().tolist())
    return []

def get_unique_books_for_grade(grade_name):
    if not grade_name or 'df_student_gap_qc' not in globals() or df_student_gap_qc.empty or 'bookName' not in df_student_gap_qc.columns:
        return []
    filtered_df = df_student_gap_qc[df_student_gap_qc['gradeName'] == grade_name]
    books = sorted(filtered_df['bookName'].unique().tolist())
    return ['All'] + books

def get_unique_chapters_for_grade_book(grade_name, book_name):
    if not grade_name or 'df_student_gap_qc' not in globals() or df_student_gap_qc.empty or 'chapterName' not in df_student_gap_qc.columns:
        return []
    filtered_df = df_student_gap_qc[df_student_gap_qc['gradeName'] == grade_name]
    if book_name and book_name != 'All':
        filtered_df = filtered_df[filtered_df['bookName'] == book_name]
    chapters = sorted(filtered_df['chapterName'].unique().tolist())
    return ['All'] + chapters

def get_unique_students_for_grade_book_chapter(grade_name, book_name, chapter_name):
    if not grade_name or 'df_student_gap_qc' not in globals() or df_student_gap_qc.empty or 'studentId' not in df_student_gap_qc.columns:
        return []
    filtered_df = df_student_gap_qc[df_student_gap_qc['gradeName'] == grade_name]
    if book_name and book_name != 'All':
        filtered_df = filtered_df[filtered_df['bookName'] == book_name]
    if chapter_name and chapter_name != 'All':
        filtered_df = filtered_df[filtered_df['chapterName'] == chapter_name]
    students = sorted(filtered_df['studentId'].unique().tolist())
    return ['All'] + [str(s) for s in students] # Convert student IDs to string for dropdown consistency


# --- Core Analysis Function ---
def analyze_and_display_results(selected_grade, selected_book, selected_chapter, selected_student):
    global all_chapters_student_final_batch_labels
    global all_chapters_students_by_batch_label
    global all_students_qc_breakdown # Use the global detailed breakdown dictionary
    global all_chapters_student_qc_label_and_weak_skills
    global all_chapters_grouped_students_by_weak_skills_and_qc

    # ANSI escape codes for colors
    COLOR_RESET = '\033[0m'
    COLOR_RED = '\033[91m'
    COLOR_GREEN = '\033[92m'
    COLOR_YELLOW = '\033[93m'
    COLOR_BLUE = '\033[94m'
    COLOR_MAGENTA = '\033[95m'
    COLOR_CYAN = '\033[96m'
    COLOR_BOLD = '\033[1m'

    # Clear previous results before running for new selections within the output widget
    clear_output(wait=True)

    all_chapters_student_final_batch_labels = {}
    all_chapters_students_by_batch_label = {}
    all_students_qc_breakdown = {} # Clear for each new analysis run
    all_chapters_student_qc_label_and_weak_skills = {} # Clear for each new analysis run
    all_chapters_grouped_students_by_weak_skills_and_qc = {} # Clear for each new analysis run


    print(f"{COLOR_BOLD}--- Running analysis for Grade: {selected_grade}, Book: {selected_book}, Chapter: {selected_chapter}, Student: {selected_student} ---{COLOR_RESET}")

    # --- Filter the main DataFrame based on selections ---
    if not selected_grade or df_student_gap_qc.empty:
        print("Please select a grade.")
        return

    filtered_analysis_df = df_student_gap_qc[df_student_gap_qc['gradeName'] == selected_grade]

    if selected_book and selected_book != 'All':
        filtered_analysis_df = filtered_analysis_df[filtered_analysis_df['bookName'] == selected_book]

    if selected_chapter and selected_chapter != 'All':
        filtered_analysis_df = filtered_analysis_df[filtered_analysis_df['chapterName'] == selected_chapter]

    # If a specific student is selected, filter further
    if selected_student and selected_student != 'All':
        try:
            student_id_int = int(selected_student)
            filtered_analysis_df = filtered_analysis_df[filtered_analysis_df['studentId'] == student_id_int]
        except ValueError:
            print(f"Invalid student ID selected: {selected_student}")
            return

    # Determine which chapters to process in the loop
    chapters_to_process = filtered_analysis_df['chapterName'].unique().tolist()
    chapters_to_process.sort()

    if not chapters_to_process or filtered_analysis_df.empty:
        print("No data found for the selected criteria.")
        return

    # Define status mapping and determine_status_label here so they are available for all sub-functions
    status_mapping = {
        'Fully Understood': 0,
        'Understands Concept, Mistake in Steps': 1,
        'Concept Not Clear': 2,
        'Not Attempted': 3,
        '': 4
    }

    def determine_status_label(qc_status_values):
        if not qc_status_values:
            return "Not Attempted"
        status_counts = Counter(qc_status_values)
        total_qcs = sum(status_counts.values())
        threshold = 0.5
        if status_counts.get(0, 0) / total_qcs > threshold:
            return "Strong"
        elif status_counts.get(1, 0) / total_qcs > threshold:
            return "Execution Issue"
        elif status_counts.get(2, 0) / total_qcs > threshold:
            return "Concept Gap"
        elif status_counts.get(3, 0) / total_qcs > threshold:
            return "Not Attempted"
        else:
            return "Mixed Gap"

    # Loop through each unique chapter name in the filtered data
    for chapter_to_filter in chapters_to_process:
        current_chapter_df = filtered_analysis_df[filtered_analysis_df['chapterName'] == chapter_to_filter]

        unique_students_in_chapter = current_chapter_df['studentId'].unique().tolist()

        if not unique_students_in_chapter:
            # Only add empty entry if we are not looking for a specific student who might not be in this particular sub-filtered chapter.
            if selected_student == 'All':
                all_chapters_student_final_batch_labels[chapter_to_filter] = {}
                all_chapters_students_by_batch_label[chapter_to_filter] = {}
            continue

        students_by_book_for_chapter = {}
        # Group by book_name within the current_chapter_df
        for book_name_inner, book_df_student in current_chapter_df.groupby('bookName'):
            students = book_df_student['studentId'].unique().tolist()
            students_by_book_for_chapter[book_name_inner] = students

        student_masterqc_by_book_chapter = {}
        for book_name_inner, student_ids_list in students_by_book_for_chapter.items():
            student_masterqc_by_book_chapter[book_name_inner] = {}
            for student_id in student_ids_list:
                filtered_df_for_masterqc = current_chapter_df[
                    (current_chapter_df['bookName'] == book_name_inner) &
                    (current_chapter_df['studentId'] == student_id)
                ]
                unique_masterqc_ids = filtered_df_for_masterqc['masterQcId'].unique().tolist()
                student_masterqc_by_book_chapter[book_name_inner][student_id] = unique_masterqc_ids

        student_qc_breakdown_by_book_chapter_current_iter = {} # Renamed to avoid confusion with previous global context
        for book_name_inner, students_in_book in student_masterqc_by_book_chapter.items():
            student_qc_breakdown_by_book_chapter_current_iter[book_name_inner] = {}
            for student_id, _ in students_in_book.items():
                student_qc_breakdown_by_book_chapter_current_iter[book_name_inner][student_id] = {}
                student_book_data_for_chapter = current_chapter_df[
                    (current_chapter_df['bookName'] == book_name_inner) &
                    (current_chapter_df['studentId'] == student_id)
                ]
                for qctype, qctype_df in student_book_data_for_chapter.groupby('qcType'):
                    qc_info_list = qctype_df[['masterQcId', 'qcStatus']].drop_duplicates().to_dict(orient='records')
                    student_qc_breakdown_by_book_chapter_current_iter[book_name_inner][student_id][qctype] = qc_info_list

        # Store the detailed breakdown for later use in 'All Students' suggestions
        all_students_qc_breakdown[chapter_to_filter] = student_qc_breakdown_by_book_chapter_current_iter

        # Continue with batch label generation (only relevant if selected_student == 'All' but we calculate it anyway)
        chapter_results_by_book = {}
        chapter_overall_student_labels_with_scores = {}

        for book_name_inner, students_data in student_qc_breakdown_by_book_chapter_current_iter.items(): # Use current_iter
            students_with_labels_and_scores_for_book = {}
            for student_id, qctype_data in students_data.items():
                base_qc_statuses = []
                add_on_qc_statuses = []
                all_qc_statuses_numerical = []

                if 'Base QC' in qctype_data:
                    for qc_info in qctype_data['Base QC']:
                        status_numerical = status_mapping.get(qc_info['qcStatus'], 4)
                        base_qc_statuses.append(status_numerical)
                        all_qc_statuses_numerical.append(status_numerical)

                if 'Add-on QC' in qctype_data:
                    for qc_info in qctype_data['Add-on QC']:
                        status_numerical = status_mapping.get(qc_info['qcStatus'], 4)
                        add_on_qc_statuses.append(status_numerical)
                        all_qc_statuses_numerical.append(status_numerical)

                base_label = determine_status_label(base_qc_statuses)
                if base_label != "Strong":
                    final_label = base_label
                else:
                    final_label = determine_status_label(add_on_qc_statuses)

                strength_score = 0.0
                if all_qc_statuses_numerical:
                    fully_understood_count = all_qc_statuses_numerical.count(0)
                    strength_score = fully_understood_count / len(all_qc_statuses_numerical)

                students_with_labels_and_scores_for_book[student_id] = {'label': final_label, 'strength_score': strength_score}
                chapter_overall_student_labels_with_scores[student_id] = {'label': final_label, 'strength_score': strength_score}

            book_students_by_batch_label_current_book = {}
            for student_id, data in students_with_labels_and_scores_for_book.items():
                label = data['label']
                strength_score = data['strength_score']
                if label not in book_students_by_batch_label_current_book:
                    book_students_by_batch_label_current_book[label] = []
                book_students_by_batch_label_current_book[label].append({'student_id': student_id, 'strength_score': strength_score})

            for label, students_list in book_students_by_batch_label_current_book.items():
                if label == 'Strong':
                    students_list.sort(key=lambda x: x['strength_score'], reverse=True)
                book_students_by_batch_label_current_book[label] = [s['student_id'] for s in students_list]

            chapter_results_by_book[book_name_inner] = book_students_by_batch_label_current_book

        all_chapters_student_final_batch_labels[chapter_to_filter] = {sid: data['label'] for sid, data in chapter_overall_student_labels_with_scores.items()}
        all_chapters_students_by_batch_label[chapter_to_filter] = chapter_results_by_book

        # --- Weak Skills Analysis and Grouping ---
        # Get students and their final QC labels for the current chapter
        current_chapter_students_qc_labels = all_chapters_student_final_batch_labels.get(chapter_to_filter, {})

        if not current_chapter_students_qc_labels:
            continue # Skip if no students in this chapter had QC labels assigned

        # 1. Aggregate Weak Skills for Students in Current Chapter
        student_aggregated_weak_skills = {}
        for student_id in current_chapter_students_qc_labels.keys():
            # Filter current_chapter_df for this student
            student_df = current_chapter_df[current_chapter_df['studentId'] == student_id]

            # Collect all weak skills entries for this student from the filtered df
            student_weak_skills_set = set()
            # Iterate over the 'weakSkills' Series for the student and handle NaN values
            for weak_skills_entry in student_df['weakSkills'].dropna():
                if isinstance(weak_skills_entry, list):
                    # If it's a list, extract skillName from each dictionary or take the string directly
                    for item in weak_skills_entry:
                        if isinstance(item, dict) and 'masterSkillName' in item:
                            student_weak_skills_set.add(item['masterSkillName'].strip())
                        elif isinstance(item, str) and item.strip():
                            student_weak_skills_set.add(item.strip())
                elif isinstance(weak_skills_entry, str) and weak_skills_entry.strip():
                    # If it's a comma-separated string, split and add
                    for ws in weak_skills_entry.split(','):
                        if ws.strip():
                            student_weak_skills_set.add(ws.strip())

            # If no weak skills identified, use a placeholder
            if not student_weak_skills_set:
                student_aggregated_weak_skills[student_id] = frozenset({'No Weak Skills Identified'})
            else:
                student_aggregated_weak_skills[student_id] = frozenset(student_weak_skills_set)

        # 2. Combine QC Label and Aggregated Weak Skills for Current Chapter
        current_chapter_student_qc_label_and_weak_skills = {}
        for student_id, qc_label in current_chapter_students_qc_labels.items():
            weak_skills_for_student = student_aggregated_weak_skills.get(student_id, frozenset({'No Weak Skills Identified'}))
            current_chapter_student_qc_label_and_weak_skills[student_id] = (qc_label, weak_skills_for_student)

        # Store this for potential future overall analysis or direct reference
        all_chapters_student_qc_label_and_weak_skills[chapter_to_filter] = current_chapter_student_qc_label_and_weak_skills

        # 3. Group Students by (QC Label, Weak Skills) Combinations for Current Chapter
        grouped_students_for_chapter = {}
        for student_id, (qc_label, weak_skills_frozenset) in current_chapter_student_qc_label_and_weak_skills.items():
            group_key = (qc_label, weak_skills_frozenset)
            if group_key not in grouped_students_for_chapter:
                grouped_students_for_chapter[group_key] = []
            grouped_students_for_chapter[group_key].append(student_id)

        # Store the grouped results per chapter
        all_chapters_grouped_students_by_weak_skills_and_qc[chapter_to_filter] = grouped_students_for_chapter


    print(f"{COLOR_BOLD}\n--- Analysis Results ---\n")

    if selected_student != 'All': # Logic for a single selected student
        if not filtered_analysis_df.empty:
            # Extract the single student's data after all filtering
            student_id_int = int(selected_student)

            # Retrieve the qctype_data using the stored breakdown
            qctype_data_for_student = None
            for chapter in chapters_to_process:
                if chapter in all_students_qc_breakdown:
                    for book_name_inner, book_breakdown in all_students_qc_breakdown[chapter].items():
                        if student_id_int in book_breakdown:
                            qctype_data_for_student = book_breakdown[student_id_int]
                            break
                if qctype_data_for_student: # Found the student's data
                    break

            if qctype_data_for_student:
                base_qc_statuses = []
                add_on_qc_statuses = []

                if 'Base QC' in qctype_data_for_student:
                    for qc_info in qctype_data_for_student['Base QC']:
                        status_numerical = status_mapping.get(qc_info['qcStatus'], 4)
                        base_qc_statuses.append(status_numerical)

                if 'Add-on QC' in qctype_data_for_student:
                    for qc_info in qctype_data_for_student['Add-on QC']:
                        status_numerical = status_mapping.get(qc_info['qcStatus'], 4)
                        add_on_qc_statuses.append(status_numerical)

                base_label = determine_status_label(base_qc_statuses)
                if base_label != "Strong":
                    student_label = base_label
                else:
                    student_label = determine_status_label(add_on_qc_statuses)

                # Get weak skills for the single selected student
                student_weak_skills_set = set()
                student_df_for_weak_skills = filtered_analysis_df[filtered_analysis_df['studentId'] == student_id_int]

                for weak_skills_entry in student_df_for_weak_skills['weakSkills'].dropna().tolist():
                    if isinstance(weak_skills_entry, list):
                        for item in weak_skills_entry:
                            if isinstance(item, dict) and 'masterSkillName' in item:
                                student_weak_skills_set.add(item['masterSkillName'].strip())
                            elif isinstance(item, str) and item.strip():
                                student_weak_skills_set.add(item.strip())
                    elif isinstance(weak_skills_entry, str) and weak_skills_entry.strip():
                        for ws in weak_skills_entry.split(','):
                            if ws.strip():
                                student_weak_skills_set.add(ws.strip())

                weak_skills_display = ", ".join(sorted(list(student_weak_skills_set))) if student_weak_skills_set else 'No Weak Skills Identified'

                print(f"{COLOR_BOLD}\n--- Analysis for Student ID: {COLOR_CYAN}{student_id_int}{COLOR_RESET} ---\n")
                print(f"Overall Assigned Batch Label: {COLOR_CYAN}{student_label}{COLOR_RESET}")
                print(f"Master Skill Name(s) (Weak Skills): [{COLOR_YELLOW}{weak_skills_display}{COLOR_RESET}]\n")

                print(f"{COLOR_BOLD}--- Suggestions for Tutors ---\n")

                # --- Specific suggestions for Base QC ---
                base_qc_type_label = determine_status_label(base_qc_statuses)
                color_base_qc = ""
                if base_qc_type_label == "Strong":
                    color_base_qc = COLOR_GREEN
                elif base_qc_type_label == "Execution Issue":
                    color_base_qc = COLOR_YELLOW
                elif base_qc_type_label == "Concept Gap":
                    color_base_qc = COLOR_RED
                elif base_qc_type_label == "Mixed Gap":
                    color_base_qc = COLOR_MAGENTA

                if base_qc_type_label != "Not Attempted": # Only provide suggestions if there are actual attempts
                    print(f"  For **Base QC** (Overall: {color_base_qc}{base_qc_type_label}{COLOR_RESET}):")
                    if base_qc_type_label == "Strong":
                        print("    - This student is strong in core concepts. Encourage them with advanced Base QC problems.")
                    elif base_qc_type_label == "Execution Issue":
                        print("    - Student understands Base QC concepts but makes errors. Focus on step-by-step practice for Base QC problems. Review common mistakes and emphasize precision.")
                    elif base_qc_type_label == "Concept Gap":
                        print("    - Student has gaps in core Base QC concepts. Revisit foundational ideas with simpler examples and provide additional teaching on prerequisite topics.")
                    elif base_qc_type_label == "Mixed Gap":
                        print("    - Base QC performance is mixed. Pinpoint specific weak areas within Base QC topics by reviewing their attempts. Offer targeted practice on those sub-topics.")
                    print("") # Blank line for readability

                # --- Specific suggestions for Add-on QC ---
                add_on_qc_type_label = determine_status_label(add_on_qc_statuses)
                color_addon_qc = ""
                if add_on_qc_type_label == "Strong":
                    color_addon_qc = COLOR_GREEN
                elif add_on_qc_type_label == "Execution Issue":
                    color_addon_qc = COLOR_YELLOW
                elif add_on_qc_type_label == "Concept Gap":
                    color_addon_qc = COLOR_RED
                elif add_on_qc_type_label == "Mixed Gap":
                    color_addon_qc = COLOR_MAGENTA

                if add_on_qc_type_label != "Not Attempted": # Only provide suggestions if there are actual attempts
                    print(f"  For **Add-on QC** (Overall: {color_addon_qc}{add_on_qc_type_label}{COLOR_RESET}):")
                    if add_on_qc_type_label == "Strong":
                        print("    - Student handles Add-on QC well. Challenge with varied and complex Add-on QC problems. Introduce them to advanced applications.")
                    elif add_on_qc_type_label == "Execution Issue":
                        print("    - Student gets Add-on QC concepts but makes mistakes. Practice careful problem-solving for Add-on QC, perhaps by working through problems together and highlighting areas for attention.")
                    elif add_on_qc_type_label == "Concept Gap":
                        print("    - Student struggles with Add-on QC concepts. Break down complex Add-on QC ideas into smaller parts. Ensure understanding of each component before moving on.")
                    elif add_on_qc_type_label == "Mixed Gap":
                        print("    - Add-on QC performance is mixed. Identify which specific Add-on QC areas need more attention. Prioritize intervention for the most significant gaps.")
                    print("") # Blank line for readability

                if base_qc_type_label == "Not Attempted" and add_on_qc_type_label == "Not Attempted":
                    print("    - This student has not tried any Base QC or Add-on QC questions. Encourage first attempts, perhaps with easier problems to build confidence.")
                print("-" * 50) # Separator for individual student
            else:
                print(f"No detailed QC status found for student {student_id_int} with the selected filters.")

        else:
            print(f"No data found for student ID {student_id_int} within the selected criteria.")

    else: # Logic for 'All' students
        if not all_chapters_students_by_batch_label:
            print("No results to display for all students.")
            return

        # First, print the batch groupings summary
        print(f"{COLOR_BOLD}---> Student IDs Grouped by Batch Label, by Chapter and by Book <---{COLOR_RESET}\n")
        for chapter, books_data in all_chapters_students_by_batch_label.items():
            print(f"{COLOR_BOLD}Chapter: {chapter}{COLOR_RESET}")
            if books_data:
                for book_name_inner, batch_labels_data in books_data.items():
                    print(f"  {COLOR_BOLD}Book Name: {book_name_inner}{COLOR_RESET}")
                    if batch_labels_data:
                        for label, student_ids in batch_labels_data.items():
                            color_label = ""
                            if label == "Strong":
                                color_label = COLOR_GREEN
                            elif label == "Execution Issue":
                                color_label = COLOR_YELLOW
                            elif label == "Concept Gap":
                                color_label = COLOR_RED
                            elif label == "Mixed Gap":
                                color_label = COLOR_MAGENTA
                            elif label == "Not Attempted":
                                color_label = COLOR_BLUE
                            print(f"    Batch Label: {color_label}{label}{COLOR_RESET}")
                            print(f"      Student IDs ({len(student_ids)}): [{', '.join([f'{COLOR_CYAN}{sid}{COLOR_RESET}' for sid in student_ids])}]")
                    else:
                        print("    No students found for this book in this chapter.")
                    print("")
                print("") # Add a blank line between books for readability
            else:
                print("  No books or students found for this chapter.")
            print("\n") # Add a blank line between chapters for readability


        # New: Display Student IDs Grouped by QC Performance Label AND Weak Skills
        print(f"{COLOR_BOLD}\n---> Student IDs Grouped by QC Performance Label AND Weak Skills, by Chapter <---{COLOR_RESET}\n")
        if not all_chapters_grouped_students_by_weak_skills_and_qc:
            print("No weak skills and QC performance groupings to display.")
        else:
            for chapter, groupings in all_chapters_grouped_students_by_weak_skills_and_qc.items():
                print(f"{COLOR_BOLD}Chapter: {chapter}{COLOR_RESET}")
                if groupings:
                    for (qc_label, weak_skills_frozenset), student_ids in groupings.items():
                        weak_skills_display = ", ".join(sorted(list(weak_skills_frozenset))) # Filter out 'No Weak Skills Identified' if actual skills exist
                        if 'No Weak Skills Identified' in weak_skills_frozenset and len(weak_skills_frozenset) > 1:
                            # If 'No Weak Skills Identified' is mistakenly included with actual skills, remove it
                            temp_skills = [s for s in weak_skills_frozenset if s != 'No Weak Skills Identified']
                            weak_skills_display = ", ".join(sorted(list(temp_skills))) if temp_skills else 'No Weak Skills Identified'
                        elif not weak_skills_display and weak_skills_frozenset == frozenset({'No Weak Skills Identified'}):
                            weak_skills_display = 'No Weak Skills Identified'
                        elif not weak_skills_display:
                            weak_skills_display = 'None'

                        color_qc_label = ""
                        if qc_label == "Strong":
                            color_qc_label = COLOR_GREEN
                        elif qc_label == "Execution Issue":
                            color_qc_label = COLOR_YELLOW
                        elif qc_label == "Concept Gap":
                            color_qc_label = COLOR_RED
                        elif qc_label == "Mixed Gap":
                            color_qc_label = COLOR_MAGENTA
                        elif qc_label == "Not Attempted":
                            color_qc_label = COLOR_BLUE

                        print(f"  QC Label: {color_qc_label}'{qc_label}'{COLOR_RESET}, Master Skill Name(s) (Weak Skills): [{COLOR_YELLOW}{weak_skills_display}{COLOR_RESET}]")
                        print(f"    Student IDs ({len(student_ids)}): [{', '.join([f'{COLOR_CYAN}{sid}{COLOR_RESET}' for sid in student_ids])}]")
                    print("")
                else:
                    print("  No students found for this chapter with weak skills and QC performance groupings.")
                print("-" * 50) # Separator for each chapter


        # Then, print detailed suggestions for each student
        print(f"{COLOR_BOLD}\n---> Detailed Suggestions for Each Student in Selected Filters <---{COLOR_RESET}\n")
        found_students_with_suggestions = False
        for chapter, books_data_for_batching in all_chapters_students_by_batch_label.items():
            if chapter in all_students_qc_breakdown:
                chapter_qc_breakdown = all_students_qc_breakdown[chapter] # Get the detailed QC data for this chapter

                for book_name_inner, batch_labels_data in books_data_for_batching.items():
                    if book_name_inner in chapter_qc_breakdown: # Check if book exists in the detailed breakdown
                        book_qc_breakdown = chapter_qc_breakdown[book_name_inner]

                        for label, student_ids_in_batch in batch_labels_data.items():
                            for student_id in student_ids_in_batch:
                                if student_id in book_qc_breakdown: # Check if student exists in the detailed breakdown for this book
                                    qctype_data_for_student = book_qc_breakdown[student_id]
                                    found_students_with_suggestions = True

                                    base_qc_statuses = []
                                    add_on_qc_statuses = []

                                    if 'Base QC' in qctype_data_for_student:
                                        for qc_info in qctype_data_for_student['Base QC']:
                                            status_numerical = status_mapping.get(qc_info['qcStatus'], 4)
                                            base_qc_statuses.append(status_numerical)

                                    if 'Add-on QC' in qctype_data_for_student:
                                        for qc_info in qctype_data_for_student['Add-on QC']:
                                            status_numerical = status_mapping.get(qc_info['qcStatus'], 4)
                                            add_on_qc_statuses.append(status_numerical)

                                    # Determine overall label for display in suggestions header
                                    base_label_for_student_sug = determine_status_label(base_qc_statuses)
                                    if base_label_for_student_sug != "Strong":
                                        overall_student_label_sug = base_label_for_student_sug
                                    else:
                                        overall_student_label_sug = determine_status_label(add_on_qc_statuses)

                                    # Retrieve weak skills for this specific student from the pre-calculated dictionary
                                    weak_skills_frozenset = frozenset({'No Weak Skills Identified'})
                                    if chapter in all_chapters_student_qc_label_and_weak_skills:
                                        if student_id in all_chapters_student_qc_label_and_weak_skills[chapter]:
                                            # The value is a tuple (qc_label, weak_skills_frozenset)
                                            _, weak_skills_frozenset = all_chapters_student_qc_label_and_weak_skills[chapter][student_id]

                                    weak_skills_display = ", ".join(sorted(list(weak_skills_frozenset))) # Filter out 'No Weak Skills Identified' if actual skills exist
                                    if 'No Weak Skills Identified' in weak_skills_frozenset and len(weak_skills_frozenset) > 1:
                                        # If 'No Weak Skills Identified' is mistakenly included with actual skills, remove it
                                        temp_skills = [s for s in weak_skills_frozenset if s != 'No Weak Skills Identified']
                                        weak_skills_display = ", ".join(sorted(list(temp_skills))) if temp_skills else 'No Weak Skills Identified'
                                    elif not weak_skills_display and weak_skills_frozenset == frozenset({'No Weak Skills Identified'}):
                                        weak_skills_display = 'No Weak Skills Identified'
                                    elif not weak_skills_display:
                                        weak_skills_display = 'None'

                                    color_overall_student_label = ""
                                    if overall_student_label_sug == "Strong":
                                        color_overall_student_label = COLOR_GREEN
                                    elif overall_student_label_sug == "Execution Issue":
                                        color_overall_student_label = COLOR_YELLOW
                                    elif overall_student_label_sug == "Concept Gap":
                                        color_overall_student_label = COLOR_RED
                                    elif overall_student_label_sug == "Mixed Gap":
                                        color_overall_student_label = COLOR_MAGENTA
                                    elif overall_student_label_sug == "Not Attempted":
                                        color_overall_student_label = COLOR_BLUE

                                    print(f"{COLOR_BOLD}--- Suggestions for Student ID: {COLOR_CYAN}{student_id}{COLOR_RESET} (Chapter: {chapter}, Book: {book_name_inner}, Overall Label: {color_overall_student_label}{overall_student_label_sug}{COLOR_RESET}, Master Skill Name(s) (Weak Skills): [{COLOR_YELLOW}{weak_skills_display}{COLOR_RESET}]){COLOR_RESET} ---")
                                    # --- Specific suggestions for Base QC ---
                                    base_qc_type_label = determine_status_label(base_qc_statuses)
                                    color_base_qc = ""
                                    if base_qc_type_label == "Strong":
                                        color_base_qc = COLOR_GREEN
                                    elif base_qc_type_label == "Execution Issue":
                                        color_base_qc = COLOR_YELLOW
                                    elif base_qc_type_label == "Concept Gap":
                                        color_base_qc = COLOR_RED
                                    elif base_qc_type_label == "Mixed Gap":
                                        color_base_qc = COLOR_MAGENTA

                                    if base_qc_type_label != "Not Attempted": # Only provide suggestions if there are actual attempts
                                        print(f"  For **Base QC** (Overall: {color_base_qc}{base_qc_type_label}{COLOR_RESET}):")
                                        if base_qc_type_label == "Strong":
                                            print("    - This student is strong in core concepts. Encourage them with advanced Base QC problems.")
                                        elif base_qc_type_label == "Execution Issue":
                                            print("    - Student understands Base QC concepts but makes errors. Focus on step-by-step practice for Base QC problems. Review common mistakes and emphasize precision.")
                                        elif base_qc_type_label == "Concept Gap":
                                            print("    - Student has gaps in core Base QC concepts. Revisit foundational ideas with simpler examples and provide additional teaching on prerequisite topics.")
                                        elif base_qc_type_label == "Mixed Gap":
                                            print("    - Base QC performance is mixed. Pinpoint specific weak areas within Base QC topics by reviewing their attempts. Offer targeted practice on those sub-topics.")
                                        print("") # Blank line for readability

                                    # --- Specific suggestions for Add-on QC ---
                                    add_on_qc_type_label = determine_status_label(add_on_qc_statuses)
                                    color_addon_qc = ""
                                    if add_on_qc_type_label == "Strong":
                                        color_addon_qc = COLOR_GREEN
                                    elif add_on_qc_type_label == "Execution Issue":
                                        color_addon_qc = COLOR_YELLOW
                                    elif add_on_qc_type_label == "Concept Gap":
                                        color_addon_qc = COLOR_RED
                                    elif add_on_qc_type_label == "Mixed Gap":
                                        color_addon_qc = COLOR_MAGENTA

                                    if add_on_qc_type_label != "Not Attempted": # Only provide suggestions if there are actual attempts
                                        print(f"  For **Add-on QC** (Overall: {color_addon_qc}{add_on_qc_type_label}{COLOR_RESET}):")
                                        if add_on_qc_type_label == "Strong":
                                            print("    - Student handles Add-on QC well. Challenge with varied and complex Add-on QC problems. Introduce them to advanced applications.")
                                        elif add_on_qc_type_label == "Execution Issue":
                                            print("    - Student gets Add-on QC concepts but makes mistakes. Practice careful problem-solving for Add-on QC, perhaps by working through problems together and highlighting areas for attention.")
                                        elif add_on_qc_type_label == "Concept Gap":
                                            print("    - Student struggles with Add-on QC concepts. Break down complex Add-on QC ideas into smaller parts. Ensure understanding of each component before moving on.")
                                        elif add_on_qc_type_label == "Mixed Gap":
                                            print("    - Add-on QC performance is mixed. Identify which specific Add-on QC areas need more attention. Prioritize intervention for the most significant gaps.")
                                        print("") # Blank line for readability

                                    if base_qc_type_label == "Not Attempted" and add_on_qc_type_label == "Not Attempted":
                                        print("    - This student has not tried any Base QC or Add-on QC questions. Encourage first attempts, perhaps with easier problems to build confidence.")
                                    print("-" * 50) # Separator for each student

        if not found_students_with_suggestions:
            print("No detailed suggestions could be generated for any student with the current filters.")

# --- Interactive widget setup ---
# 1. Create dropdown widgets
grade_options = get_unique_grades()
grade_dropdown = widgets.Dropdown(
    options=grade_options,
    value=grade_options[0] if grade_options else None,
    description='Select Grade:',
    disabled=False,
)

initial_book_options = get_unique_books_for_grade(grade_dropdown.value)
book_dropdown = widgets.Dropdown(
    options=initial_book_options,
    value=initial_book_options[0] if initial_book_options else None,
    description='Select Book:',
    disabled=False,
)

initial_chapter_options = get_unique_chapters_for_grade_book(grade_dropdown.value, book_dropdown.value)
chapter_dropdown = widgets.Dropdown(
    options=initial_chapter_options,
    value=initial_chapter_options[0] if initial_chapter_options else None,
    description='Select Chapter:',
    disabled=False,
)

initial_student_options = get_unique_students_for_grade_book_chapter(grade_dropdown.value, book_dropdown.value, chapter_dropdown.value)
student_dropdown = widgets.Dropdown(
    options=initial_student_options,
    value=initial_student_options[0] if initial_student_options else None,
    description='Select Student:',
    disabled=False,
)

# 2. Define observer functions to update options of subsequent dropdowns
def update_book_and_chapter_and_student_dropdowns(change):
    selected_grade = change.new

    book_dropdown.value = None # Clear value first
    new_book_options = get_unique_books_for_grade(selected_grade)
    book_dropdown.options = new_book_options
    book_dropdown.value = new_book_options[0] if new_book_options else None

    chapter_dropdown.value = None # Clear value first
    new_chapter_options = get_unique_chapters_for_grade_book(selected_grade, book_dropdown.value)
    chapter_dropdown.options = new_chapter_options
    chapter_dropdown.value = new_chapter_options[0] if new_chapter_options else None

    student_dropdown.value = None # Clear value first
    new_student_options = get_unique_students_for_grade_book_chapter(selected_grade, book_dropdown.value, chapter_dropdown.value)
    student_dropdown.options = new_student_options
    student_dropdown.value = new_student_options[0] if new_student_options else None

def update_chapter_and_student_dropdowns_from_book(change):
    selected_grade = grade_dropdown.value
    selected_book = change.new

    chapter_dropdown.value = None # Clear value first
    new_chapter_options = get_unique_chapters_for_grade_book(selected_grade, selected_book)
    chapter_dropdown.options = new_chapter_options
    chapter_dropdown.value = new_chapter_options[0] if new_chapter_options else None

    student_dropdown.value = None # Clear value first
    new_student_options = get_unique_students_for_grade_book_chapter(selected_grade, selected_book, chapter_dropdown.value)
    student_dropdown.options = new_student_options
    student_dropdown.value = new_student_options[0] if new_student_options else None

def update_student_dropdown_only_from_chapter(change):
    selected_grade = grade_dropdown.value
    selected_book = book_dropdown.value
    selected_chapter = change.new

    student_dropdown.value = None # Clear value first
    new_student_options = get_unique_students_for_grade_book_chapter(selected_grade, selected_book, selected_chapter)
    student_dropdown.options = new_student_options
    student_dropdown.value = new_student_options[0] if new_student_options else None

# 3. Link dropdowns using observe
grade_dropdown.observe(update_book_and_chapter_and_student_dropdowns, names='value')
book_dropdown.observe(update_chapter_and_student_dropdowns_from_book, names='value')
chapter_dropdown.observe(update_student_dropdown_only_from_chapter, names='value')

# 4. Create an Output widget to capture the analysis results
analysis_output_widget = widgets.Output()

# 5. Create a Run Analysis button
run_button = widgets.Button(
    description='INSIGHT GENERATOR',
    disabled=False,
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click to run the analysis for selected criteria'
)

# 6. Define the function to call when the button is clicked
def on_button_clicked(b):
    with analysis_output_widget:
        analyze_and_display_results(grade_dropdown.value, book_dropdown.value, chapter_dropdown.value, student_dropdown.value)

# 7. Link the button to the on_button_clicked function
run_button.on_click(on_button_clicked)

# 8. Display all widgets, the button, and the output area
print("Please select the Grade, Book, Chapter, and Student to analyze:")
display(grade_dropdown, book_dropdown, chapter_dropdown, student_dropdown, run_button, analysis_output_widget)